In [56]:
# Loading libraries

import os
import pandas as pd
from astropy.table import Table

In [ ]:
#catalog = 'glimpse2_v2cat'
catalog = 'glimpse_s07'

In [58]:
alma_regions = Table.read("../alma_input/almaprops_2024_all_20250219.tab", format="ascii.ipac")
print(alma_regions[:2])
print("\n\n\n")

    ALMAGAL_ID           ORIG_ID        INTERNAL_ID ... FIT_FLAG OBS_SAMPLE
                                                    ...                    
----------------- --------------------- ----------- ... -------- ----------
AG010.1313-0.7687 HIGALBM10.1330-0.7710       46677 ...        -       near
AG010.1392-0.7831 HIGALBM10.1395-0.7833       46696 ...        -       near






In [59]:
spitzer_sources = Table.read("../output/tables/spitzer_sources_"+catalog+".tbl", format="ascii.ipac")
print(spitzer_sources[:2])
print("\n\n\n")

cntr_01  dist_x    pang_x   internal_id_01 ... color45_58 color45_80 color58_80
         arcsec     deg                    ...                                 
------- -------- ---------- -------------- ... ---------- ---------- ----------
      1 2.947689 -55.310982          46677 ...      0.529      0.361     -0.168
      1 4.216898  84.415197          46677 ...         --         --         --






In [60]:
alma_sources = pd.read_csv('../alma_input/7MTM2TM1_Core_catalogue_out_official_v3_run6Apr23+16Jan24_sn-5_may24_clump_cat.txt', comment='\\', skiprows=[2], delimiter='\s+')
print(alma_sources[6001:6004])
print("\n\n\n")

      ID               CLUMP          X          Y         RA        DEC  \
6001  22  G323.7399-00.2617B  386.92831  335.97809  232.93886 -56.514006   
6002  23  G323.7399-00.2617B  374.90641  340.99945  232.93932 -56.513900   
6003  20  G323.7399-00.2617B  355.04385  326.97879  232.94008 -56.514196   

           GLON      GLAT     F_PEAK         S/N  ...  Back_onpeak_jybm  \
6001  323.73984 -0.262544  1489.3764   31.570764  ...          0.004688   
6002  323.74011 -0.262602  6066.5668  128.594859  ...          0.006919   
6003  323.74028 -0.263085  1283.1450   27.199214  ...          0.011580   

            Rcore  Surfd_core        n(H2)              AG_ID  \
6001   984.988790    3.902623   63200000.0  AG323.7408-0.2633   
6002  1044.259595   15.896265  243000000.0  AG323.7408-0.2633   
6003   973.958180    3.362233   55100000.0  AG323.7408-0.2633   

                 ORIG_ID  RADIO_INFO  RADIO_MATCH  Surfd_nd VLSR_clump  
6001  G323.7399-00.2617B           1            0      1.58 

In [61]:
os.makedirs('../output/alma_sources', exist_ok=True)

for region in alma_regions:

    alma_sources_cut = alma_sources[alma_sources['CLUMP'] == region['INTERNAL_ID']].reset_index(drop=True) # is a pandas object
    spitzer_sources_cut = spitzer_sources[spitzer_sources['internal_id_01'] == region['INTERNAL_ID']] # is a Table object

    os.makedirs(f'../output/alma_sources/{region["INTERNAL_ID"]}', exist_ok=True)
    
    fwhm_circ = 0.5

    if len(alma_sources_cut) > 0:    
        with open(f'../output/alma_sources/{region["INTERNAL_ID"]}/{region["INTERNAL_ID"]}_alma_sources.reg', 'w') as f:
            f.write('# Region file format: DS9 version 4.1\n')
            f.write('global color=green dashlist=8 3 width=1 font="helvetica 10 normal roman" select=1 highlite=1 dash=0 fixed=0 edit=1 move=1 delete=1 include=1 source=1\n')
            f.write('icrs\n')
            
            for alma_index in range(len(alma_sources_cut)):
                f.write('circle('+str(alma_sources_cut['RA'][alma_index])+','+str(alma_sources_cut['DEC'][alma_index])+','+str(fwhm_circ/3600)+') # text={'+str(alma_index+1)+'}\n')


    if len(spitzer_sources_cut) > 0:
        with open(f'../output/alma_sources/{region["INTERNAL_ID"]}/{region["INTERNAL_ID"]}_{catalog}_spitzer_sources.reg', 'w') as f:
            f.write('# Region file format: DS9 version 4.1\n')
            f.write('global color=red dashlist=8 3 width=1 font="helvetica 10 normal roman" select=1 highlite=1 dash=0 fixed=0 edit=1 move=1 delete=1 include=1 source=1\n')
            f.write('icrs\n')

            for spitzer_index in range(len(spitzer_sources_cut)):
                f.write('circle('+str(spitzer_sources_cut['ra'][spitzer_index])+','+str(spitzer_sources_cut['dec'][spitzer_index])+','+str(fwhm_circ/3600)+')\n')
                